# Data Exploration
Distributions, class balance, and lab vs PPMI feature sanity-check.
No training here — see `train.py`.

In [ ]:
import sys, warnings
sys.path.insert(0, '.')
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from src.data.lab import load_lab
from src.data.ppmi import load_ppmi
from src.data.normalize import aseg_common_features

lab_df, lab_aseg_cols, lab_thick_cols = load_lab()
ppmi_df, ppmi_aseg_cols = load_ppmi()
common_aseg = aseg_common_features()

print(f"Lab  : {len(lab_df)} subjects | columns: {lab_df.shape[1]}")
print(f"PPMI : {len(ppmi_df)} subjects | columns: {ppmi_df.shape[1]}")
print(f"Common ASEG features: {len(common_aseg)}")


## Class Balance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4))

for ax, (name, df) in zip(axes, [("Lab", lab_df), ("PPMI", ppmi_df)]):
    counts = df['label'].value_counts().sort_index()
    bars = ax.bar(['HC', 'PD'], counts.values, color=['#1565c0', '#c62828'], alpha=0.85, width=0.5)
    for bar, v in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.5, str(v),
                ha='center', va='bottom', fontweight='bold', fontsize=11)
    ax.set_title(f'{name} — Class Balance', fontsize=13, fontweight='bold')
    ax.set_ylabel('Number of subjects')
    ax.spines[['top','right']].set_visible(False)
    total = counts.sum()
    ax.set_ylim(0, counts.max() * 1.2)
    pct = counts / total * 100
    ax.set_xlabel(f"HC: {pct[0]:.1f}%  |  PD: {pct[1]:.1f}%  |  Total: {total}")

plt.suptitle('Class Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('class_balance.png', dpi=150, bbox_inches='tight')
plt.show()


## Age & Sex Distributions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for col_i, (name, df) in enumerate([("Lab", lab_df), ("PPMI", ppmi_df)]):
    # Age
    ax = axes[0, col_i]
    for label_val, color, grp in [(0, '#1565c0', 'HC'), (1, '#c62828', 'PD')]:
        ages = df.loc[df['label'] == label_val, 'Age'].dropna()
        ax.hist(ages, bins=15, alpha=0.65, color=color, label=f'{grp} (n={len(ages)})')
    ax.set_title(f'{name} — Age Distribution', fontweight='bold')
    ax.set_xlabel('Age (years)')
    ax.set_ylabel('Count')
    ax.legend()
    ax.spines[['top','right']].set_visible(False)

    # Sex
    ax = axes[1, col_i]
    for label_val, color, grp in [(0, '#1565c0', 'HC'), (1, '#c62828', 'PD')]:
        sub = df[df['label'] == label_val]
        n_male   = int(sub['Sex'].sum())
        n_female = len(sub) - n_male
        ax.bar([f'{grp}\nMale', f'{grp}\nFemale'], [n_male, n_female],
               color=[color]*2, alpha=[0.85, 0.5], width=0.4)
    ax.set_title(f'{name} — Sex Distribution', fontweight='bold')
    ax.set_ylabel('Count')
    ax.spines[['top','right']].set_visible(False)

plt.suptitle('Age & Sex by Group', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('demographics.png', dpi=150, bbox_inches='tight')
plt.show()


## Lab vs PPMI Feature Sanity Check
For a handful of key ASEG features, compare mean ± std in PD and HC across both datasets. If the canonical column mapping is correct, the value ranges should be similar.

In [ ]:
key_features = [
    'Left_Putamen', 'Right_Putamen',
    'Left_Caudate', 'Right_Caudate',
    'Left_Hippocampus', 'Right_Hippocampus',
    'Left_Thalamus', 'Right_Thalamus',
    'TotalGrayVol', 'EstimatedTotalIntraCranialVol',
]
key_features = [f for f in key_features if f in lab_df.columns and f in ppmi_df.columns]

rows = []
for feat in key_features:
    for name, df in [('Lab', lab_df), ('PPMI', ppmi_df)]:
        for grp_val, grp_name in [(0,'HC'),(1,'PD')]:
            vals = df.loc[df['label']==grp_val, feat].dropna()
            rows.append({'Dataset':name,'Group':grp_name,'Feature':feat,
                         'Mean':vals.mean(),'Std':vals.std(),'N':len(vals)})

summary = pd.DataFrame(rows)
pivot = summary.pivot_table(index=['Feature','Group'], columns='Dataset',
                             values='Mean').round(0)
print(pivot.to_string())


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(20, 7))
axes = axes.flatten()

for ax, feat in zip(axes, key_features):
    for col_i, (name, df) in enumerate([('Lab', lab_df), ('PPMI', ppmi_df)]):
        for j, (label_val, color, grp) in enumerate([(0,'#1565c0','HC'),(1,'#c62828','PD')]):
            vals = df.loc[df['label']==label_val, feat].dropna()
            x = col_i * 2 + j
            ax.bar(x, vals.mean(), yerr=vals.std(), color=color, alpha=0.7 if col_i==0 else 0.4,
                   width=0.7, capsize=4)
    ax.set_xticks([0,1,2,3])
    ax.set_xticklabels(['Lab HC','Lab PD','PPMI HC','PPMI PD'], rotation=30, ha='right', fontsize=7)
    ax.set_title(feat.replace('_',' '), fontsize=8, fontweight='bold')
    ax.spines[['top','right']].set_visible(False)

fig.suptitle('Mean ± SD of Key ASEG Features: Lab vs PPMI', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('feature_sanity.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: class_balance.png, demographics.png, feature_sanity.png")
